# v3.4 — pembersih detak: penyelarasan R-peak ke templat QRS + gerbang morfologi

Rancangan dikunci sebelum hasil: [RANCANGAN.md](RANCANGAN.md) (sha256 di `../logs/v3.4_praregistrasi.log`). Logika di `pembersih.py`.

- **F1**: setiap R-peak dari detektor produksi digeser (≤ 3 sampel) ke posisi yang paling cocok dengan templat QRS median ±5 s.
- **F2**: F1 + window ditolak ("sinyal buruk", tanpa alarm) bila < 60% detaknya mirip templat.

Ide dari pembaca EKG Tugas Akhir pemilik repo (`asd_ecg/measurement.py`, `_dominant_beats`). Detektor produksi tidak diubah;
paket produksi tidak disentuh.

In [1]:
import json, subprocess, sys, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import pembersih as P
sys.path[:0] = ["../v11_model_tim"]
from bersama import evaluasi as Ev
pd.set_option("display.width", 200)
R = {}   # ringkasan
t_mulai = time.time()

## 1. Bukti: tes, front-end lama identik, uji paritas training

In [2]:
out = subprocess.run([sys.executable, "-m", "pytest", "test_pembersih.py", "-q"], capture_output=True, text=True)
print(out.stdout[-400:]); assert out.returncode == 0
d_f1 = P.bangun_afdb()      # meng-assert front-end lama = cache/dataset_v2_detector.npz (bit demi bit)
z_lama = np.load(P.DATA_LAMA)
R["window_afdb"] = {"lama": np.bincount(z_lama["y"], minlength=3).tolist(), "f1": np.bincount(d_f1["y"], minlength=3).tolist()}
print("window per kelas (Normal, AFIB, AFL):", R["window_afdb"])
R["paritas"] = P.cek_paritas()
print("uji paritas (fungsi job v3.4 pada dataset lama vs OOF MicroTransformer_r0 v3):", R["paritas"])
assert R["paritas"]["argmax_sama"] >= 0.999

.......                                                                  [100%]
7 passed in 2.27s

window per kelas (Normal, AFIB, AFL): {'lama': [99000, 65826, 1152], 'f1': [98962, 65872, 1152]}
uji paritas (fungsi job v3.4 pada dataset lama vs OOF MicroTransformer_r0 v3): {'selisih_maks': 0.0, 'argmax_sama': 1.0, 'n': 165978}


## 2. Front-end: galat posisi R-peak di AFDB (vs anotasi `.qrs`) dan gerbang

In [3]:
g = np.load(P.CACHE / "galat_afdb.npz")
recs = P.daftar_rekaman()
rows = {}
for kode, nama in ((0, "Normal"), (1, "AFIB"), (2, "AFL")):
    r = {}
    for fe in ("lama", "f1"):
        e = np.concatenate([g[f"{x}__galat_{fe}"][g[f"{x}__ritme_{fe}"] == kode] for x in recs])
        e = e[np.abs(e) <= 100]          # buang pasangan meleset (> 100 ms setelah offset) agar sd tidak didominasi ekor
        r[f"sd galat {fe} (ms)"] = float(np.std(e)); r[f"|galat| > 20 ms {fe}"] = float(np.mean(np.abs(e) > 20))
    rows[nama] = r
R["galat_afdb"] = rows
dom = np.array([g[f"{x}__dominan_detak"] for x in recs])
R["detak_dominan_afdb"] = {"median per rekaman": float(np.median(dom[:, 0])), "min": float(dom[:, 0].min())}
lama, f1 = P.data_afdb()
R["gerbang_afdb"] = {P.CLASSES[c]: float(f1["lolos_gerbang"][f1["y"] == c].mean()) for c in range(3)}
R["cv_median_afdb"] = {P.CLASSES[c]: {"lama": float(np.median(lama["cv"][lama["y"] == c])), "f1": float(np.median(f1["cv"][f1["y"] == c]))}
                       for c in range(3)}
display(pd.DataFrame(rows).T.round(3)); print("window lolos gerbang F2:", R["gerbang_afdb"]); print("CV IBI median:", R["cv_median_afdb"])

,sd galat lama (ms),|galat| > 20 ms lama,sd galat f1 (ms),|galat| > 20 ms f1
Normal,16.555,0.158,15.067,0.129
AFIB,18.442,0.174,18.407,0.172
AFL,11.995,0.070,12.116,0.067


window lolos gerbang F2: {'Normal': 0.9874194135122573, 'AFIB': 0.9820712897741074, 'AFL': 1.0}
CV IBI median: {'Normal': {'lama': 0.025184210389852524, 'f1': 0.019132748246192932}, 'AFIB': {'lama': 0.2134997546672821, 'f1': 0.21304547786712646}, 'AFL': {'lama': 0.03048263117671013, 'f1': 0.030168049037456512}}


## 3. H1 — AFDB: model F1 (+bias) vs v3.1, protokol butir 1–5

Bootstrap pasien 2.000× (seed 2026, strata dari dataset lama), berpasangan dengan bobot pasien yang sama. Window kedua dataset
berbeda (front-end berbeda), tetapi pasiennya sama, sehingga confusion matrix per pasien bisa dibandingkan.

In [4]:
W = P.bobot_bootstrap(lama["strata"])
c_v31, c_f1, c_f2 = P.cms(lama, lama["pred"]), P.cms(f1, f1["pred"]), P.cms(f1, f1["pred_f2"])
c_v1 = P.cms(lama, lama["v1"].argmax(-1))
met, boot, titik = {}, {}, {}
for k, c in (("v31", c_v31), ("f1", c_f1), ("f2", c_f2), ("v1", c_v1)):
    met[k], boot[k] = P.ringkas_metrik(c, W)
    titik[k] = {m: v["nilai"] for m, v in met[k].items()}
R["metrik_afdb"] = met
R["banding_afdb"] = {"f1_vs_v31": P.banding(boot["f1"], boot["v31"], titik["f1"], titik["v31"]),
                     "f2_vs_v31": P.banding(boot["f2"], boot["v31"], titik["f2"], titik["v31"]),
                     "f2_vs_f1": P.banding(boot["f2"], boot["f1"], titik["f2"], titik["f1"]),
                     "f1_vs_v1": P.banding(boot["f1"], boot["v1"], titik["f1"], titik["v1"])}
tab = pd.DataFrame({k: {m: f"{v['nilai']:.3f} [{v['ci'][0]:.2f}–{v['ci'][1]:.2f}]" for m, v in met[k].items()} for k in met}).T
display(tab)
for nama, b in R["banding_afdb"].items():
    print(nama, {m: f"{v['delta']:+.3f} [{v['ci'][0]:+.3f}; {v['ci'][1]:+.3f}] P={v['p']:.3f} {v['vonis']}" for m, v in b.items()})

,Macro-F1,Normal R,AFIB P,AFIB R,AFIB F1,AFL P,AFL R,AFL F1
v31,0.832 [0.67–0.92],0.924 [0.84–0.97],0.887 [0.80–0.94],0.932 [0.90–0.96],0.909 [0.86–0.94],0.715 [0.26–0.95],0.592 [0.13–0.91],0.647 [0.17–0.89]
f1,0.842 [0.68–0.93],0.938 [0.89–0.97],0.907 [0.85–0.95],0.941 [0.91–0.97],0.924 [0.90–0.95],0.712 [0.20–0.93],0.606 [0.13–0.92],0.654 [0.16–0.89]
f2,0.840 [0.68–0.93],0.944 [0.90–0.97],0.913 [0.86–0.96],0.925 [0.90–0.95],0.919 [0.89–0.94],0.713 [0.20–0.94],0.606 [0.13–0.92],0.654 [0.16–0.89]
v1,0.744 [0.64–0.81],0.903 [0.82–0.95],0.869 [0.79–0.93],0.929 [0.90–0.95],0.898 [0.85–0.93],0.275 [0.05–0.46],0.728 [0.37–0.98],0.399 [0.09–0.59]


f1_vs_v31 {'Macro-F1': '+0.011 [-0.002; +0.021] P=0.962 lebih baik', 'AFL R': '+0.014 [-0.020; +0.020] P=0.891 tidak terbedakan', 'AFL F1': '+0.007 [-0.030; +0.015] P=0.788 tidak terbedakan', 'AFIB R': '+0.009 [+0.001; +0.017] P=0.989 lebih baik', 'Normal R': '+0.014 [-0.009; +0.048] P=0.824 tidak terbedakan'}
f2_vs_v31 {'Macro-F1': '+0.009 [-0.006; +0.020] P=0.886 tidak terbedakan', 'AFL R': '+0.014 [-0.020; +0.020] P=0.891 tidak terbedakan', 'AFL F1': '+0.007 [-0.029; +0.015] P=0.796 tidak terbedakan', 'AFIB R': '-0.007 [-0.026; +0.010] P=0.267 tidak terbedakan', 'Normal R': '+0.020 [-0.006; +0.059] P=0.898 tidak terbedakan'}
f2_vs_f1 {'Macro-F1': '-0.002 [-0.007; +0.002] P=0.208 tidak terbedakan', 'AFL R': '+0.000 [+0.000; +0.000] P=0.000 identik', 'AFL F1': '+0.000 [+0.000; +0.001] P=0.644 tidak terbedakan', 'AFIB R': '-0.016 [-0.032; -0.003] P=0.000 lebih buruk', 'Normal R': '+0.006 [+0.001; +0.012] P=1.000 lebih baik'}
f1_vs_v1 {'Macro-F1': '+0.099 [+0.027; +0.171] P=0.991 lebih 

In [5]:
R["afl_per_pasien"] = {k: P.afl_per_pasien(d, p) for k, (d, p) in
                       {"v31": (lama, lama["pred"]), "f1": (f1, f1["pred"]), "f2": (f1, f1["pred_f2"])}.items()}
R["afl_palsu"] = {"v31": P.asal_afl_palsu(lama, lama["pred"]), "f1": P.asal_afl_palsu(f1, f1["pred"]),
                  "f2": P.asal_afl_palsu(f1, f1["pred_f2"])}
display(pd.DataFrame({k: {p: f"{v['recall']:.2f} (n={v['n']})" for p, v in d.items()} for k, d in R["afl_per_pasien"].items()}))
display(pd.DataFrame(R["afl_palsu"]).round(1))
R["bias_f1"] = {"median": np.median(f1["bias"].reshape(-1, 3), 0).tolist(),
                "di_batas_grid": float(np.mean((f1["bias"][..., 1:] <= -7) | (f1["bias"][..., 1:] >= 3)))}
print("bias cross-fit F1 (median [Normal, AFIB, AFL]):", R["bias_f1"])

,v31,f1,f2
04043,0.40 (n=1),0.40 (n=1),0.40 (n=1)
04908,0.92 (n=46),0.87 (n=46),0.87 (n=46)
04936,0.51 (n=673),0.53 (n=673),0.53 (n=673)
06426,0.17 (n=21),0.20 (n=21),0.20 (n=21)
07910,0.00 (n=96),0.00 (n=96),0.00 (n=96)
08215,0.00 (n=9),0.00 (n=9),0.00 (n=9)
08378,0.96 (n=306),0.97 (n=306),0.97 (n=306)


,v31,f1,f2
"Normal, pasien tanpa AFL",4.0,15.2,14.6
"Normal, pasien AFL",51.4,49.6,49.6
"AFIB, pasien tanpa AFL",21.4,19.0,19.0
"AFIB, pasien AFL",197.4,200.2,200.2


bias cross-fit F1 (median [Normal, AFIB, AFL]): {'median': [0.0, -0.75, -4.5], 'di_batas_grid': 0.0}


### Metrik episode (butir 7), window Normal cepat (butir 9), dan window yang sama di kedua dataset

In [6]:
R["episode"] = {"v31": P.episode(lama, lama["pred"]), "f1": P.episode(f1, f1["pred"]), "f2": P.episode(f1, f1["pred_f2"])}
for kls in ("AFIB", "AFL"):
    print(kls); display(pd.DataFrame({k: v[kls] for k, v in R["episode"].items()}).round(3))
hb = P.T.by_hr_bin(f1, {"f1": f1["pred"], "f2": f1["pred_f2"]})
hb_l = P.T.by_hr_bin(lama, {"v31": lama["pred"]})
R["per_pita_afdb"] = {k: {"n_f1": hb[k]["n"], "f1": hb[k]["f1"], "f2": hb[k]["f2"], "n_lama": hb_l[k]["n"], "v31": hb_l[k]["v31"]} for k in hb}
display(pd.DataFrame({k: {"n lama": v["n_lama"], "v3.1 % benar": v["v31"][P.CLASSES.index(k.split(' |')[0])],
                          "n F1": v["n_f1"], "F1 % benar": v["f1"][P.CLASSES.index(k.split(' |')[0])]} for k, v in R["per_pita_afdb"].items()}).T.round(3))
il, i1 = P.window_sama(lama, f1)
pl = P.prediksi_subset(lama, il, lama["probs"]["v31"]); p1 = P.prediksi_subset(f1, i1, f1["probs"])
sl, s1 = P.subset(lama, il, pl), P.subset(f1, i1, p1)
ml, bl = P.ringkas_metrik(P.cms(sl, pl), W); m1, b1 = P.ringkas_metrik(P.cms(s1, p1), W)
R["window_sama"] = {"n": int(len(il)), "fraksi_lama": float(len(il) / len(lama["y"])), "v31": ml, "f1": m1,
                    "banding": P.banding(b1, bl, {m: v["nilai"] for m, v in m1.items()}, {m: v["nilai"] for m, v in ml.items()})}
print("window sama:", R["window_sama"]["n"], {m: f"{v['delta']:+.3f} P={v['p']:.2f}" for m, v in R["window_sama"]["banding"].items()})

AFIB


,v31,f1,f2
se_episode,0.970,0.972,0.936
ppv_episode,0.562,0.562,0.603
se_durasi,0.932,0.941,0.925
ppv_durasi,0.887,0.907,0.913
n_ref,881.000,856.000,856.000
n_det,6252.400,5466.200,5658.000
palsu_per_jam,11.858,10.356,9.729


AFL


,v31,f1,f2
se_episode,0.686,0.657,0.657
ppv_episode,0.403,0.380,0.381
se_durasi,0.592,0.606,0.606
ppv_durasi,0.715,0.712,0.713
n_ref,14.000,14.000,14.000
n_det,198.200,198.400,197.800
palsu_per_jam,0.521,0.536,0.534


,n lama,v3.1 % benar,n F1,F1 % benar
Normal | 0–100 bpm,89566.0,0.933,89529.0,0.944
Normal | 100–110 bpm,8766.0,0.873,8781.0,0.913
Normal | 110–120 bpm,568.0,0.410,548.0,0.488
Normal | 120–∞ bpm,100.0,0.048,104.0,0.056
AFIB | 0–100 bpm,46038.0,0.916,46060.0,0.927
AFIB | 100–110 bpm,10716.0,0.964,10706.0,0.974
AFIB | 110–120 bpm,5045.0,0.986,5026.0,0.988
AFIB | 120–∞ bpm,4027.0,0.953,4080.0,0.954
AFL | 0–100 bpm,156.0,0.000,156.0,0.000
AFL | 100–110 bpm,27.0,0.000,27.0,0.000


window sama: 154422 {'Macro-F1': '+0.008 P=0.96', 'AFL R': '+0.010 P=0.70', 'AFL F1': '+0.009 P=0.84', 'AFIB R': '+0.005 P=0.94', 'Normal R': '+0.009 P=0.83'}


### Protokol 10: bootstrap bersarang dan tanpa 04936

In [7]:
cand_l = P.T.cached_predictions(lama)[2]   # kandidat bias v3.1 dari cache v3.2 (probabilitas sama)
cand_1 = P.kandidat_cms(f1)
s_l, s_1 = P.seri_bersarang(lama, cand_l, W), P.seri_bersarang(f1, cand_1, W)
assert abs(s_l[0] - titik["v31"]["Macro-F1"]) < 1e-9 and abs(s_1[0] - titik["f1"]["Macro-F1"]) < 1e-9
dd = s_1[1:] - s_l[1:]
R["bersarang"] = {"v31": [float(s_l[0]), *np.percentile(s_l[1:], [2.5, 97.5])], "f1": [float(s_1[0]), *np.percentile(s_1[1:], [2.5, 97.5])],
                  "delta": float(s_1[0] - s_l[0]), "ci": np.percentile(dd, [2.5, 97.5]).tolist(), "p": float(np.mean(dd > 0))}
print("bersarang:", R["bersarang"])
keep_l, keep_1 = lama["rec"] != P.T.SUSPECT, f1["rec"] != P.T.SUSPECT
il0, i10 = np.flatnonzero(keep_l), np.flatnonzero(keep_1)
pl0, p10 = P.prediksi_subset(lama, il0, lama["probs"]["v31"]), P.prediksi_subset(f1, i10, f1["probs"])
pats = np.array([p for p in lama["patients"] if p != P.T.SUSPECT])
sl0 = dict(P.subset(lama, il0, pl0), patients=pats); s10 = dict(P.subset(f1, i10, p10), patients=pats)
W0 = P.cv.bootstrap_weights(P.cv.patient_strata(sl0["y"], sl0["rec"], pats), P.N_BOOT, seed=P.SEED)
ml0, bl0 = P.ringkas_metrik(P.cms(sl0, pl0), W0); m10, b10 = P.ringkas_metrik(P.cms(s10, p10), W0)
R["tanpa_04936"] = {"v31": ml0, "f1": m10, "banding": P.banding(b10, bl0, {m: v["nilai"] for m, v in m10.items()},
                                                                 {m: v["nilai"] for m, v in ml0.items()})}
print("tanpa 04936:", {m: f"{v['delta']:+.3f} P={v['p']:.2f}" for m, v in R["tanpa_04936"]["banding"].items()})

bersarang: {'v31': [0.8315489481663656, np.float64(0.6595510543143901), np.float64(0.9101562911559574)], 'f1': [0.8422557271488709, np.float64(0.6758347635893659), np.float64(0.9213849879033225)], 'delta': 0.010706778982505272, 'ci': [-0.0035133640388776114, 0.03615481898535947], 'p': 0.9455}


tanpa 04936: {'Macro-F1': '+0.012 P=1.00', 'AFL R': '+0.025 P=1.00', 'AFL F1': '+0.007 P=0.85', 'AFIB R': '+0.013 P=1.00', 'Normal R': '+0.015 P=0.86'}


## 4. H5 — drop-in: model yang dilatih dengan front-end lama, diuji di window F1

Model fold `v7 kontrol` (resep v5b, ulangan 0, bobot tersimpan), bias fold dari OOF lamanya. Dibandingkan dengan OOF-nya sendiri
di window lama. Menjawab: bisakah bobot paket v3.1 dipakai ulang dengan front-end baru tanpa latih ulang?

In [8]:
pk_lama, pk_f1 = P.prediksi_dropin(f1, device="cuda" if P.cv.torch.cuda.is_available() else "cpu")
dl = dict(y=lama["y"], rec=lama["rec"], patients=lama["patients"]); d1 = dict(y=f1["y"], rec=f1["rec"], patients=f1["patients"])
ml_, bl_ = P.ringkas_metrik(P.cms(dl, pk_lama), W); m1_, b1_ = P.ringkas_metrik(P.cms(d1, pk_f1), W)
R["dropin"] = {"lama": ml_, "f1": m1_, "banding": P.banding(b1_, bl_, {m: v["nilai"] for m, v in m1_.items()},
                                                             {m: v["nilai"] for m, v in ml_.items()})}
print({m: f"{v['delta']:+.3f} [{v['ci'][0]:+.3f}; {v['ci'][1]:+.3f}] P={v['p']:.2f} {v['vonis']}" for m, v in R["dropin"]["banding"].items()})

{'Macro-F1': '+0.005 [-0.002; +0.013] P=0.92 tidak terbedakan', 'AFL R': '+0.004 [+0.000; +0.020] P=0.90 tidak terbedakan', 'AFL F1': '-0.006 [-0.029; +0.003] P=0.07 tidak terbedakan', 'AFIB R': '-0.001 [-0.004; +0.003] P=0.36 tidak terbedakan', 'Normal R': '+0.017 [+0.003; +0.044] P=1.00 lebih baik'}


## 5. Data latihan fisik (v3.3): H2, H3, H4

Grid window sama dengan v3.3 (10 s, tidak tumpang tindih). Front-end lama harus mereproduksi `cache/v3_3/jendela.npz`. Himpunan
window untuk uji berpasangan ditetapkan dengan detak front-end lama; window gagal cek kualitas = tidak ada alarm.

In [9]:
L = P.jendela_latihan()
print("window identik dengan v3.3:", P.cek_latihan_v33(L))
K = P.prediksi_latihan(L, device="cuda" if P.cv.torch.cuda.is_available() else "cpu")
subj = L["subjek"]
data = {}
for s in ("EPFL", "Wrist", "EPHNOGRAM"):
    m = L["sumber"] == s
    data[s] = {"window": int(m.sum()), "subjek": int(np.unique(subj[m]).size)}
    for fe in ("lama", "f1"):
        ok = m & L[f"ok_{fe}"]
        data[s][f"lolos QC {fe}"] = float(ok.sum() / m.sum()); data[s][f"CV median {fe}"] = float(np.nanmedian(L[f"cv_{fe}"][ok]))
    data[s]["lolos gerbang F2 (dari lolos QC F1)"] = float((L["dominan"][m & L["ok_f1"]] >= P.MIN_DOMINAN).mean())
gm = {}
for s in ("EPFL", "Wrist"):
    m = L["galat_sumber"] == s
    gm[s] = {"jitter lama (ms, median rekaman)": float(np.nanmedian(L["galat_lama"][m])), "jitter F1": float(np.nanmedian(L["galat_f1"][m])),
             "rekaman membaik": float(np.mean(L["galat_f1"][m] < L["galat_lama"][m]))}
R["latihan_data"], R["latihan_jitter"] = data, gm
display(pd.DataFrame(data).T.round(3)); display(pd.DataFrame(gm).T.round(3))

window identik dengan v3.3: 11872


,window,subjek,lolos QC lama,CV median lama,lolos QC f1,CV median f1,lolos gerbang F2 (dari lolos QC F1)
EPFL,198.0,20.0,0.682,0.028,0.899,0.012,0.989
Wrist,670.0,8.0,0.836,0.066,0.851,0.060,0.793
EPHNOGRAM,11004.0,24.0,0.920,0.047,0.922,0.040,0.940


,"jitter lama (ms, median rekaman)",jitter F1,rekaman membaik
EPFL,10.685,2.841,0.768
Wrist,14.669,10.411,0.474


In [10]:
a_l, a_1, a_2 = (K["v31_lama"] > 0).astype(float), (K["v31_f1"] > 0).astype(float), (K["v31_f2"] > 0).astype(float)
h2 = np.isin(L["sumber"], ["Wrist", "EPHNOGRAM"]) & (L["hr_lama"] < 100)
R["H2"] = {"alarm_lama": float(a_l[h2].mean()), "alarm_f1": float(a_1[h2].mean()), **P.boot_subjek_selisih(subj[h2], a_1[h2] - a_l[h2])}
R["H2_per_sumber"] = {s: {"lama": float(a_l[h2 & (L["sumber"] == s)].mean()), "f1": float(a_1[h2 & (L["sumber"] == s)].mean()),
                          "f2": float(a_2[h2 & (L["sumber"] == s)].mean())} for s in ("Wrist", "EPHNOGRAM")}
rule_l = P.T33.kena_aturan(L["hr_lama"], L["cv_lama"]); rule_1 = P.T33.kena_aturan(L["hr_f1"], L["cv_f1"])
h3 = L["hr_lama"] >= 120
sisa_l, sisa_1, sisa_2 = a_l * ~rule_l, a_1 * ~rule_1, a_2 * ~rule_1
R["H3b"] = {"sisa_lama": float(sisa_l[h3].mean()), "sisa_f1": float(sisa_1[h3].mean()), **P.boot_subjek_selisih(subj[h3], sisa_1[h3] - sisa_l[h3])}
R["H3b_per_sumber"] = {s: {"n": int((h3 & (L["sumber"] == s)).sum()), "alarm lama": float(a_l[h3 & (L["sumber"] == s)].mean()),
                           "alarm F1": float(a_1[h3 & (L["sumber"] == s)].mean()),
                           "sisa lama": float(sisa_l[h3 & (L["sumber"] == s)].mean()), "sisa F1": float(sisa_1[h3 & (L["sumber"] == s)].mean()),
                           "sisa F2": float(sisa_2[h3 & (L["sumber"] == s)].mean())} for s in ("EPFL", "Wrist", "EPHNOGRAM")}
m_e = L["sumber"] == "EPFL"
qc = pd.DataFrame({"lama": pd.Series(L["ok_lama"][m_e]).groupby(subj[m_e]).mean(), "f1": pd.Series(L["ok_f1"][m_e]).groupby(subj[m_e]).mean()})
R["H3a"] = {"lolos_lama": float(L["ok_lama"][m_e].mean()), "lolos_f1": float(L["ok_f1"][m_e].mean()),
            "subjek_naik": int((qc["f1"] > qc["lama"]).sum()), "subjek_turun": int((qc["f1"] < qc["lama"]).sum()), "subjek": int(len(qc))}
R["H4_aktivitas"] = {"alarm_f1": float(a_1[h2].mean()), "alarm_f2": float(a_2[h2].mean()), **P.boot_subjek_selisih(subj[h2], a_2[h2] - a_1[h2])}
for k in ("H2", "H3a", "H3b", "H4_aktivitas"): print(k, R[k])
display(pd.DataFrame(R["H2_per_sumber"]).T.round(3)); display(pd.DataFrame(R["H3b_per_sumber"]).T.round(3))

H2 {'alarm_lama': 0.20356145251396648, 'alarm_f1': 0.17702513966480446, 'delta': -0.02653631284916201, 'ci': [-0.05300008316893244, -0.009203962214518021], 'p_kurang': 1.0, 'n': 5728, 'subjek': 32}
H3a {'lolos_lama': 0.6818181818181818, 'lolos_f1': 0.898989898989899, 'subjek_naik': 15, 'subjek_turun': 0, 'subjek': 20}
H3b {'sisa_lama': 0.30023006134969327, 'sisa_f1': 0.2331288343558282, 'delta': -0.06710122699386503, 'ci': [-0.14666103683314918, -0.017372889110623627], 'p_kurang': 1.0, 'n': 2608, 'subjek': 42}
H4_aktivitas {'alarm_f1': 0.17702513966480446, 'alarm_f2': 0.11958798882681564, 'delta': -0.057437150837988823, 'ci': [-0.10598046580522538, -0.021185995790176753], 'p_kurang': 1.0, 'n': 5728, 'subjek': 32}


,lama,f1,f2
Wrist,0.259,0.224,0.087
EPHNOGRAM,0.200,0.174,0.122


,n,alarm lama,alarm F1,sisa lama,sisa F1,sisa F2
EPFL,195.0,0.682,0.903,0.185,0.103,0.103
Wrist,107.0,0.617,0.617,0.103,0.084,0.047
EPHNOGRAM,2306.0,0.848,0.852,0.319,0.251,0.235


In [11]:
# alarm per pita detak (detak front-end sendiri, hanya window lolos QC; pola v3.3) dan alarm sinus takikardia ≥ 120 bpm
pita = {}
for s in ("EPFL", "Wrist", "EPHNOGRAM"):
    for nama, kls, fe in (("v3.1 lama", K["v31_lama"], "lama"), ("v3.1 + F1", K["v31_f1"], "f1"), ("v3.1 + F2", K["v31_f2"], "f1"),
                          ("fold F1", K["fold_f1"], "f1")):
        ok = (L["sumber"] == s) & L[f"ok_{fe}"]
        al = np.atleast_2d(kls)[:, ok]; hr = L[f"hr_{fe}"][ok]
        pita.setdefault(s, {})[nama] = {f"{lo}–{hi if hi < 999 else '∞'}": [int(((hr >= lo) & (hr < hi)).sum()), float((al[:, (hr >= lo) & (hr < hi)] > 0).mean())]
                                         for lo, hi in P.T33.PITA if ((hr >= lo) & (hr < hi)).any()}
R["alarm_pita"] = pita
utama = {}
for nama, kls, fe in (("v3.1 lama", K["v31_lama"], "lama"), ("v3.1 + F1", K["v31_f1"], "f1"), ("v3.1 + F2", K["v31_f2"], "f1"),
                      ("fold F1", K["fold_f1"], "f1")):
    ok = L[f"ok_{fe}"] & (L[f"hr_{fe}"] >= 120)
    t_, ci, _ = P.T33._boot_subjek(subj[ok], (np.atleast_2d(kls)[:, ok] > 0).mean(0))
    utama[nama] = {"n": int(ok.sum()), "alarm": t_, "ci": ci.tolist()}
R["alarm_120"] = utama
for s in pita: print("==", s); display(pd.DataFrame({k: {b: f"{v[1]*100:.0f}% (n={v[0]})" for b, v in d.items()} for k, d in pita[s].items()}).T)
display(pd.DataFrame(utama).T)

== EPFL


,0–100,130–140,140–150,150–∞
v3.1 lama,0% (n=2),100% (n=3),100% (n=17),100% (n=113)
v3.1 + F1,50% (n=2),100% (n=3),100% (n=19),100% (n=154)
v3.1 + F2,0% (n=2),100% (n=3),100% (n=19),100% (n=154)
fold F1,78% (n=2),100% (n=3),98% (n=19),100% (n=154)


== Wrist


,0–100,100–110,110–120,120–130,130–140,140–150,150–∞
v3.1 lama,29% (n=305),22% (n=108),63% (n=81),100% (n=64),100% (n=2),NaN,NaN
v3.1 + F1,25% (n=307),20% (n=108),57% (n=82),93% (n=67),0% (n=1),67% (n=3),100% (n=2)
v3.1 + F2,10% (n=307),14% (n=108),50% (n=82),90% (n=67),0% (n=1),33% (n=3),50% (n=2)
fold F1,28% (n=307),17% (n=108),47% (n=82),91% (n=67),64% (n=1),87% (n=3),100% (n=2)


== EPHNOGRAM


,0–100,100–110,110–120,120–130,130–140,140–150,150–∞
v3.1 lama,21% (n=5034),16% (n=1862),62% (n=1241),97% (n=813),99% (n=439),99% (n=278),100% (n=460)
v3.1 + F1,19% (n=5033),12% (n=1863),59% (n=1244),95% (n=812),99% (n=433),99% (n=288),100% (n=478)
v3.1 + F2,13% (n=5033),9% (n=1863),55% (n=1244),93% (n=812),97% (n=433),95% (n=288),100% (n=478)
fold F1,18% (n=5033),9% (n=1863),40% (n=1244),90% (n=812),98% (n=433),98% (n=288),99% (n=478)


,n,alarm,ci
v3.1 lama,2189,0.984011,"[0.965077, 0.995563]"
v3.1 + F1,2260,0.975664,"[0.951888, 0.989869]"
v3.1 + F2,2260,0.95708,"[0.917895, 0.981567]"
fold F1,2260,0.950336,"[0.922932, 0.970232]"


## 6. Data eksternal (amandemen A): paket v3.1 2 kelas, front-end lama vs F1 vs F2

CinC 2017, MIT-BIH, LTAFDB, CPSC 2021 (label resmi). Grid window `signal_to_windows` sama untuk semua front-end; window gagal cek
kualitas = tidak ada alarm. Bootstrap subjek berpasangan 2.000× (`v16.1.deteksi_af.banding2_pasien`). H6: sensitivitas F1 (dan F2)
vs front-end lama bukan "lebih buruk" di setiap dataset.

In [12]:
R163, U162 = P._modul_eksternal()
A161 = U162.A161
eks = {}
for ds in P.EKSTERNAL:
    Z = P.muat_eksternal(ds)
    y2 = Z["y2"].astype(np.int64)
    pl, p1 = Z["v31_lama"] > 0, Z["v31_f1"] > 0
    p2 = p1 & (Z["dominan"] >= P.MIN_DOMINAN)
    e = {"window": int(len(y2)), "AF": int(y2.sum()), "subjek": int(np.unique(Z["subjek"]).size),
         "subjek_AF": int(np.unique(Z["subjek"][y2 == 1]).size),
         "lolos_qc": {fe: {"non-AF": float(Z[f"ok_{fe}"][y2 == 0].mean()), "AF": float(Z[f"ok_{fe}"][y2 == 1].mean())} for fe in ("lama", "f1")},
         "lolos_gerbang": {"non-AF": float((Z["dominan"][(y2 == 0) & Z["ok_f1"]] >= P.MIN_DOMINAN).mean()),
                           "AF": float((Z["dominan"][(y2 == 1) & Z["ok_f1"]] >= P.MIN_DOMINAN).mean())},
         "f1_vs_lama": A161.banding2_pasien(y2, p1, pl, Z["subjek"]), "f2_vs_lama": A161.banding2_pasien(y2, p2, pl, Z["subjek"])}
    both = Z["ok_lama"] & Z["ok_f1"]
    e["qc_keduanya"] = {"n": int(both.sum()), "f1_vs_lama": A161.banding2_pasien(y2[both], p1[both], pl[both], Z["subjek"][both])}
    pita = {}
    for lo, hi in ((0, 100), (100, 110), (110, 120), (120, 999)):
        m = (y2 == 0) & (Z["hr_lama"] >= lo) & (Z["hr_lama"] < hi)
        pita[f"{lo}–{hi if hi < 999 else '∞'}"] = {"n": int(m.sum()), "lama": float(pl[m].mean()) if m.any() else None,
                                                   "f1": float(p1[m].mean()) if m.any() else None, "f2": float(p2[m].mean()) if m.any() else None}
    e["alarm_pita_nonaf"] = pita
    # per subjek ber-AF (penyimpangan B, deskriptif): Se lama/F1/F2 dan fraksi window AF yang ditolak gerbang
    ps = {}
    for sub in np.unique(Z["subjek"][y2 == 1]):
        m = (Z["subjek"] == sub) & (y2 == 1)
        ps[str(sub)] = {"n": int(m.sum()), "lama": float(pl[m].mean()), "f1": float(p1[m].mean()), "f2": float(p2[m].mean()),
                        "ditolak_gerbang": float(((Z["dominan"] < P.MIN_DOMINAN) & Z["ok_f1"])[m].mean())}
    e["per_subjek_af"] = ps
    e["subjek_turun_0_10"] = {fe: sorted([s_ for s_, v in ps.items() if v[fe] < v["lama"] - 0.10]) for fe in ("f1", "f2")}
    eks[ds] = e
R["eksternal"] = eks
rows = {}
for ds, e in eks.items():
    for nama, b in (("F1", e["f1_vs_lama"]), ("F2", e["f2_vs_lama"])):
        rows[f"{ds} / {nama}"] = {m: f"{b['titik_b'][m][0]:.3f} → {b['titik_a'][m][0]:.3f} (Δ {b[m][0]:+.3f}, P = {b[m][3]:.3f})" for m in ("Se", "Sp", "Macro-F1")}
display(pd.DataFrame(rows).T)
print("subjek ber-AF dengan Se turun > 0,10:", {ds: e["subjek_turun_0_10"] for ds, e in eks.items()})
display(pd.DataFrame(eks["MIT-BIH"]["per_subjek_af"]).T.round(3))
display(pd.DataFrame({ds: {f"{b} {fe}": (f"{v[fe]*100:.1f}% (n={v['n']})" if v[fe] is not None else "–") for b, v in e["alarm_pita_nonaf"].items()
                           for fe in ("lama", "f1", "f2")} for ds, e in eks.items()}))

,Se,Sp,Macro-F1
CinC 2017 / F1,"0.845 → 0.846 (Δ +0.001, P = 0.610)","0.890 → 0.902 (Δ +0.012, P = 1.000)","0.753 → 0.769 (Δ +0.016, P = 1.000)"
CinC 2017 / F2,"0.845 → 0.834 (Δ -0.011, P = 0.003)","0.890 → 0.908 (Δ +0.018, P = 1.000)","0.753 → 0.774 (Δ +0.021, P = 1.000)"
MIT-BIH / F1,"0.904 → 0.896 (Δ -0.007, P = 0.038)","0.860 → 0.872 (Δ +0.012, P = 1.000)","0.753 → 0.764 (Δ +0.011, P = 0.998)"
MIT-BIH / F2,"0.904 → 0.888 (Δ -0.016, P = 0.000)","0.860 → 0.882 (Δ +0.022, P = 1.000)","0.753 → 0.774 (Δ +0.021, P = 0.999)"
LTAFDB / F1,"0.916 → 0.916 (Δ -0.001, P = 0.164)","0.919 → 0.934 (Δ +0.016, P = 1.000)","0.917 → 0.923 (Δ +0.007, P = 1.000)"
LTAFDB / F2,"0.916 → 0.905 (Δ -0.011, P = 0.000)","0.919 → 0.948 (Δ +0.030, P = 1.000)","0.917 → 0.924 (Δ +0.007, P = 0.977)"
CPSC 2021 / F1,"0.906 → 0.906 (Δ -0.000, P = 0.446)","0.862 → 0.872 (Δ +0.010, P = 1.000)","0.869 → 0.875 (Δ +0.006, P = 1.000)"
CPSC 2021 / F2,"0.906 → 0.875 (Δ -0.031, P = 0.000)","0.862 → 0.883 (Δ +0.021, P = 1.000)","0.869 → 0.870 (Δ +0.002, P = 0.646)"


subjek ber-AF dengan Se turun > 0,10: {'CinC 2017': {'f1': ['A00132', 'A00680', 'A00885', 'A01079', 'A01302', 'A01309', 'A01784', 'A01811', 'A02165', 'A02587', 'A03760', 'A04048', 'A04604', 'A05366', 'A06251', 'A06288', 'A06932', 'A07168', 'A07243', 'A07258', 'A07518', 'A07771', 'A07822', 'A08405', 'A08451'], 'f2': ['A00015', 'A00132', 'A00643', 'A00680', 'A00722', 'A00837', 'A00885', 'A00982', 'A01079', 'A01302', 'A01309', 'A01359', 'A01711', 'A01784', 'A01811', 'A02070', 'A02165', 'A02289', 'A02587', 'A03606', 'A03760', 'A03852', 'A04048', 'A04604', 'A04675', 'A04704', 'A04943', 'A05366', 'A05703', 'A05989', 'A06251', 'A06288', 'A06925', 'A06932', 'A07168', 'A07243', 'A07253', 'A07258', 'A07518', 'A07771', 'A07810', 'A07822', 'A08405', 'A08414', 'A08451']}, 'MIT-BIH': {'f1': [], 'f2': ['222']}, 'LTAFDB': {'f1': [], 'f2': ['122', '16', '56']}, 'CPSC 2021': {'f1': [], 'f2': ['25', '33', '58', '70', '81']}}


,n,lama,f1,f2,ditolak_gerbang
201,231.0,0.970,0.970,0.970,0.000
203,314.0,0.783,0.780,0.771,0.016
210,339.0,0.929,0.903,0.903,0.000
217,18.0,0.889,0.833,0.833,0.000
219,267.0,0.891,0.891,0.891,0.000
221,327.0,0.951,0.948,0.948,0.000
222,39.0,0.949,0.974,0.718,0.282


,CinC 2017,MIT-BIH,LTAFDB,CPSC 2021
0–100 lama,9.2% (n=38696),11.8% (n=11827),7.1% (n=555010),13.4% (n=216591)
0–100 f1,8.0% (n=38696),10.6% (n=11827),5.5% (n=555010),12.3% (n=216591)
0–100 f2,7.4% (n=38696),9.4% (n=11827),4.7% (n=555010),11.3% (n=216591)
100–110 lama,16.6% (n=1033),18.8% (n=656),23.5% (n=11965),16.1% (n=5184)
100–110 f1,12.9% (n=1033),17.8% (n=656),16.3% (n=11965),15.9% (n=5184)
100–110 f2,11.9% (n=1033),17.8% (n=656),5.3% (n=11965),13.8% (n=5184)
110–120 lama,51.7% (n=468),75.8% (n=351),36.9% (n=8414),38.8% (n=1990)
110–120 f1,48.3% (n=468),75.5% (n=351),35.0% (n=8414),39.6% (n=1990)
110–120 f2,46.4% (n=468),75.5% (n=351),18.1% (n=8414),34.8% (n=1990)
120–∞ lama,72.0% (n=768),78.3% (n=23),18.4% (n=16402),66.8% (n=638)


## 7. Uji stres takikardia sintetis (butir 9) untuk model fold F1

In [13]:
st = P.stres_fold_f1(f1)
R["stres"] = {j: {"ambang": P.T.normal_threshold(tb), "normal": tb[:, 0].tolist()} for j, tb in st.items()}
R["stres_bpm"] = P.T.BPMS
print({j: v["ambang"] for j, v in R["stres"].items()}, "| referensi v3.1 paket: 115 bpm")

{'tanpa variasi': 120, 'seragam ±10 ms': 120, 'Gauss sd 10 ms': 115, 'Gauss sd 20 ms': 115} | referensi v3.1 paket: 115 bpm


## 8. Keputusan (aturan RANCANGAN bagian 5 + amandemen A)

In [14]:
b = R["banding_afdb"]["f1_vs_v31"]
H1 = b["Macro-F1"]["p"] > 0.05 and b["AFL R"]["p"] > 0.05
H2 = R["H2"]["p_kurang"] >= 0.95
H3b = R["H3b"]["p_kurang"] >= 0.95
H5 = R["dropin"]["banding"]["Macro-F1"]["p"] > 0.05 and R["dropin"]["banding"]["AFL R"]["p"] > 0.05
biaya = {c: titik["f1"][f"{c} R"] - titik["f2"][f"{c} R"] for c in ("AFIB", "AFL")}
H4 = (R["H4_aktivitas"]["p_kurang"] >= 0.95 and all(v <= 0.02 for v in biaya.values())
      and R["banding_afdb"]["f2_vs_v31"]["Macro-F1"]["p"] > 0.05)
H6_f1 = all(e["f1_vs_lama"]["Se"][3] > 0.05 for e in R["eksternal"].values())
H6_f2 = all(e["f2_vs_lama"]["Se"][3] > 0.05 for e in R["eksternal"].values())
F1_rek = H1 and (H2 or H3b) and H5 and H6_f1
R["keputusan"] = {"H1": H1, "H2": H2, "H3b": H3b, "H4": H4, "H5": H5, "biaya_f2_recall": biaya,
                  "H6_f1": H6_f1, "H6_f2": H6_f2,
                  "F1_direkomendasikan": F1_rek, "F2_direkomendasikan": F1_rek and H4 and H6_f2}
print(json.dumps(R["keputusan"], indent=1, default=float))

{
 "H1": true,
 "H2": true,
 "H3b": true,
 "H4": true,
 "H5": true,
 "biaya_f2_recall": {
  "AFIB": 0.016365071654117158,
  "AFL": 0.0
 },
 "H6_f1": false,
 "H6_f2": false,
 "F1_direkomendasikan": false,
 "F2_direkomendasikan": false
}


## 9. Kesimpulan

In [15]:
m = R["metrik_afdb"]; bb = R["banding_afdb"]["f1_vs_v31"]; k = R["keputusan"]
print(f"""AFDB (H1): Macro-F1 v3.1 {m['v31']['Macro-F1']['nilai']:.3f} vs F1 {m['f1']['Macro-F1']['nilai']:.3f} (Δ {bb['Macro-F1']['delta']:+.3f}, P = {bb['Macro-F1']['p']:.2f}, {bb['Macro-F1']['vonis']});
  recall AFL {m['v31']['AFL R']['nilai']:.3f} vs {m['f1']['AFL R']['nilai']:.3f} (P = {bb['AFL R']['p']:.2f}, {bb['AFL R']['vonis']}). H1 {'lolos' if k['H1'] else 'GAGAL'}.
Aktivitas < 100 bpm (H2): alarm {R['H2']['alarm_lama']*100:.1f}% → {R['H2']['alarm_f1']*100:.1f}% (P(Δ<0) = {R['H2']['p_kurang']:.3f}). H2 {'lolos' if k['H2'] else 'gagal'}.
Sisa alarm ≥ 120 bpm sesudah aturan produk (H3b): {R['H3b']['sisa_lama']*100:.1f}% → {R['H3b']['sisa_f1']*100:.1f}% (P = {R['H3b']['p_kurang']:.3f}). H3b {'lolos' if k['H3b'] else 'gagal'}.
EPFL lolos QC (H3a): {R['H3a']['lolos_lama']*100:.0f}% → {R['H3a']['lolos_f1']*100:.0f}%.
Drop-in (H5): Δ Macro-F1 {R['dropin']['banding']['Macro-F1']['delta']:+.3f} (P = {R['dropin']['banding']['Macro-F1']['p']:.2f}). H5 {'lolos' if k['H5'] else 'gagal'}.
Gerbang F2 (H4): alarm < 100 bpm {R['H4_aktivitas']['alarm_f1']*100:.1f}% → {R['H4_aktivitas']['alarm_f2']*100:.1f}% (P = {R['H4_aktivitas']['p_kurang']:.3f}); biaya recall AFIB {biaya['AFIB']:+.3f}, AFL {biaya['AFL']:+.3f}. H4 {'lolos' if k['H4'] else 'gagal'}.
Bersarang: Δ {R['bersarang']['delta']:+.3f} [{R['bersarang']['ci'][0]:+.3f}; {R['bersarang']['ci'][1]:+.3f}], P = {R['bersarang']['p']:.2f}. Tanpa 04936: Δ {R['tanpa_04936']['banding']['Macro-F1']['delta']:+.3f}, P = {R['tanpa_04936']['banding']['Macro-F1']['p']:.2f}.
Data eksternal (H6, Se F1 vs lama): {', '.join(f"{ds} {e['f1_vs_lama']['Se'][0]:+.3f} (P = {e['f1_vs_lama']['Se'][3]:.2f})" for ds, e in R['eksternal'].items())}.
  F2: {', '.join(f"{ds} {e['f2_vs_lama']['Se'][0]:+.3f} (P = {e['f2_vs_lama']['Se'][3]:.2f})" for ds, e in R['eksternal'].items())}. H6 F1 {'lolos' if k['H6_f1'] else 'GAGAL'}, F2 {'lolos' if k['H6_f2'] else 'GAGAL'}.
Uji stres: ambang {R['stres']['tanpa variasi']['ambang']} bpm (v3.1: 115).
Keputusan: F1 {'DIREKOMENDASIKAN' if k['F1_direkomendasikan'] else 'tidak direkomendasikan'}; F2 {'direkomendasikan' if k['F2_direkomendasikan'] else 'tidak direkomendasikan'}.""")

AFDB (H1): Macro-F1 v3.1 0.832 vs F1 0.842 (Δ +0.011, P = 0.96, lebih baik);
  recall AFL 0.592 vs 0.606 (P = 0.89, tidak terbedakan). H1 lolos.
Aktivitas < 100 bpm (H2): alarm 20.4% → 17.7% (P(Δ<0) = 1.000). H2 lolos.
Sisa alarm ≥ 120 bpm sesudah aturan produk (H3b): 30.0% → 23.3% (P = 1.000). H3b lolos.
EPFL lolos QC (H3a): 68% → 90%.
Drop-in (H5): Δ Macro-F1 +0.005 (P = 0.92). H5 lolos.
Gerbang F2 (H4): alarm < 100 bpm 17.7% → 12.0% (P = 1.000); biaya recall AFIB +0.016, AFL +0.000. H4 lolos.
Bersarang: Δ +0.011 [-0.004; +0.036], P = 0.95. Tanpa 04936: Δ +0.012, P = 1.00.
Data eksternal (H6, Se F1 vs lama): CinC 2017 +0.001 (P = 0.61), MIT-BIH -0.007 (P = 0.04), LTAFDB -0.001 (P = 0.16), CPSC 2021 -0.000 (P = 0.45).
  F2: CinC 2017 -0.011 (P = 0.00), MIT-BIH -0.016 (P = 0.00), LTAFDB -0.011 (P = 0.00), CPSC 2021 -0.031 (P = 0.00). H6 F1 GAGAL, F2 GAGAL.
Uji stres: ambang 120 bpm (v3.1: 115).
Keputusan: F1 tidak direkomendasikan; F2 tidak direkomendasikan.


In [16]:
R["lingkungan"] = Ev._lingkungan(); R["durasi_notebook_menit"] = (time.time() - t_mulai) / 60
P.simpan_json(R, P.CACHE / "ringkasan.json"); print("tersimpan:", P.CACHE / "ringkasan.json")

tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v3_4/ringkasan.json
